In [1]:
import argparse
import logging
import json
from pathlib import Path

In [2]:
import matplotlib
matplotlib.use("Agg")   # безголовый режим (нет GUI) — всегда работает на сервере
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

In [3]:
from config.settings import (
    COLOR_NEGATIVE,
    COLOR_NEUTRAL,
    COLOR_POSITIVE,
    COLOR_PUBLIC,
    COLOR_STATE,
    EVENTS,
    FIGURE_DPI,
    FIGURE_SIZE_SQUARE,
    FIGURE_SIZE_WIDE,
    FIGURES_DIR,
    SENTIMENT_LABEL_NAMES,
    RESULTS_DIR, 
    CLUSTERS_CSV, 
    METRICS_JSON,
)

log = logging.getLogger(__name__)

ModuleNotFoundError: No module named 'config'

In [ ]:
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "figure.dpi":        FIGURE_DPI,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
})

NameError: name 'FIGURE_DPI' is not defined

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", palette="muted")

In [ ]:
def _add_event_markers(
    ax: plt.Axes,
    events: list[dict] = EVENTS,
    ymin: float = 0.0,
    ymax: float = 1.0,
    color: str = "#e67e22",
    alpha: float = 0.7,
) -> None:
    """
    Добавляет вертикальные линии-маркеры ключевых событий на оси ax.
    Метки событий подписываются наклонно над линией.
    """
    for ev in events:
        x = pd.Timestamp(ev["date"], tz="UTC")
        ax.axvline(x=x, color=color, linestyle="--", linewidth=1.0, alpha=alpha)
        ax.text(
            x, ymax * 0.97,
            ev["short"],
            rotation=90,
            fontsize=7,
            color=color,
            alpha=0.85,
            va="top",
            ha="right",
            bbox={
                "boxstyle": "round,pad=0.15",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.65,
            },
        )

In [ ]:
def _save(fig: plt.Figure, name: str, show: bool = False) -> Path:
    """Сохраняет фигуру в FIGURES_DIR и опционально показывает."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / f"{name}.png"
    bbox_inches = None if fig.get_constrained_layout() else "tight"
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches=bbox_inches)
    log.info("График сохранён → %s", path)
    if show:
        plt.show()
    plt.close(fig)
    return path


In [ ]:
def plot_activity_timeline(
    df: pd.DataFrame,
    show: bool = False,
) -> Path:
    """
    Число публикаций по неделям с маркерами событий.
    df должен содержать: period (datetime), n_posts.
    """
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_WIDE)

    df2 = df.copy()
    df2["period"] = pd.to_datetime(df2["period"])

    ax.bar(df2["period"], df2["n_posts"],
           width=5, color="#3498db", alpha=0.75, label="Число постов")
    ax.set_xlabel("Дата")
    ax.set_ylabel("Число публикаций")
    ax.set_title("Рис. 2.1. Динамика публикационной активности в Telegram-каналах\n"
                 "(март–декабрь 2025 г., недельная агрегация)")

    y_max = df2["n_posts"].max() * 1.15
    _add_event_markers(ax, ymax=y_max)
    ax.set_ylim(0, y_max)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=30, ha="right")
    ax.legend()
    fig.tight_layout()

    return _save(fig, "fig_2_1_activity_timeline", show)

In [ ]:
def plot_sentiment_timeline(
    df: pd.DataFrame,
    show: bool = False,
) -> Path:
    """
    Стекированная диаграмма долей тональности по неделям.
    df: period, pct_positive, pct_neutral, pct_negative.
    """
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_WIDE)

    df2 = df.copy()
    df2["period"] = pd.to_datetime(df2["period"])
    df2 = df2.sort_values("period")

    ax.stackplot(
        df2["period"],
        df2["pct_positive"],
        df2["pct_neutral"],
        df2["pct_negative"],
        labels=["Позитивная", "Нейтральная", "Негативная"],
        colors=[COLOR_POSITIVE, COLOR_NEUTRAL, COLOR_NEGATIVE],
        alpha=0.75,
    )

    _add_event_markers(ax, ymax=100)
    ax.set_xlabel("Дата")
    ax.set_ylabel("Доля публикаций, %")
    ax.set_ylim(0, 100)
    ax.set_title("Рис. 2.2. Динамика тональности публикаций в Telegram-каналах\n"
                 "(март–декабрь 2025 г., недельная агрегация)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=30, ha="right")
    ax.legend(loc="upper left")
    fig.tight_layout()

    return _save(fig, "fig_2_2_sentiment_timeline", show)